# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
#Instalar cliente de Groq
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from google.colab import userdata

In [ ]:
# y leer API key desde Colab Secrets


client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"
# prompt = "Explica en un párrafo qué hace un router doméstico"

print(prompt)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
#comprobar modelos de la cuenta
models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
groq/compound-mini
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
whisper-large-v3
groq/compound
allam-2-7b
openai/gpt-oss-safeguard-20b
qwen/qwen3.6-27b


In [ ]:
#provamos con otro modelo
response = client.chat.completions.create(
    model="openai/gpt-oss-20b", #<--- usamos penai
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

En una computadora hay dos tipos de memoria que cumplen funciones muy distintas: **la RAM (memoria de acceso aleatorio)** y el **almacenamiento (también llamado memoria secundaria o de almacenamiento permanente)**.  
A continuación se explica cada una y qué las diferencia:

| Característica | RAM (memoria principal) | Almacenamiento (memoria secundaria) |
|----------------|-------------------------|-------------------------------------|
| **Función** | Guarda los datos y programas que la CPU necesita usar **en ese instante**. | Guarda de manera permanente los datos, el sistema operativo, las aplicaciones y todos tus archivos. |
| **Persistencia** | **Volátil**: se borra cuando el equipo se apaga o reinicia. | **No volátil**: mantiene la información incluso cuando está apagado. |
| **Velocidad** | Muy rápida (GB/s en el rango de 10‑20 GB/s, con latencias de pocos microsegundos). | Más lenta (especialmente los discos duros mecánicos; los SSD son más rápidos, pero siguen siendo ~10‑50 × m

“Nuestra conexión con Groq sí funciona. Podemos comprobarlo porque Groq nos devuelve su catálogo de modelos. El problema es que el modelo Llama utilizado cuando se creó esta práctica ya no está habilitado en nuestra cuenta. Esto es algo normal cuando trabajamos con APIs: los proveedores pueden actualizar su catálogo de modelos.”

In [ ]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "id": "chatcmpl-93a0d0f9-463b-4d73-9df3-08b650e24241",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "En una computadora hay dos tipos de memoria que cumplen funciones muy distintas: **la RAM (memoria de acceso aleatorio)** y el **almacenamiento (también llamado memoria secundaria o de almacenamiento permanente)**.  \nA continuación se explica cada una y qué las diferencia:\n\n| Característica | RAM (memoria principal) | Almacenamiento (memoria secundaria) |\n|----------------|-------------------------|-------------------------------------|\n| **Función** | Guarda los datos y programas que la CPU necesita usar **en ese instante**. | Guarda de manera permanente los datos, el sistema operativo, las aplicaciones y todos tus archivos. |\n| **Persistencia** | **Volátil**: se borra cuando el equipo se apaga o reinicia. | **No volátil**: mantiene la información incluso cuando está apagado. |\n| **Velocid

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print("Tokens del prompt:", response.usage.prompt_tokens)
print("Tokens de la respuesta:", response.usage.completion_tokens)
print("Tokens totales:", response.usage.total_tokens)
# response.usage.total_tokens es lo que se factura por esta llamada

Tokens del prompt: 86
Tokens de la respuesta: 1242
Tokens totales: 1328


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

import time

inicio = time.time()
response_tiempo = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 0.81 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()

response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}]
)

duracion_grande = time.time() - inicio

print(f"Modelo ligero (GPT-OSS 20B): {duracion:.2f} s — {response.usage.total_tokens} tokens")
print(f"Modelo grande (GPT-OSS 120B): {duracion_grande:.2f} s — {response_grande.usage.total_tokens} tokens")

# ver ambas respuesta
print("\n==RESPUESTA DEL MODELO CHICO==\n")
print(response.choices[0].message.content)

print("\n=RESPUESTA DEL MODELO GRANDE==\n")
print(response_grande.choices[0].message.content)


Modelo ligero (GPT-OSS 20B): 0.81 s — 1328 tokens
Modelo grande (GPT-OSS 120B): 2.74 s — 1333 tokens

==RESPUESTA DEL MODELO CHICO==

En una computadora hay dos tipos de memoria que cumplen funciones muy distintas: **la RAM (memoria de acceso aleatorio)** y el **almacenamiento (también llamado memoria secundaria o de almacenamiento permanente)**.  
A continuación se explica cada una y qué las diferencia:

| Característica | RAM (memoria principal) | Almacenamiento (memoria secundaria) |
|----------------|-------------------------|-------------------------------------|
| **Función** | Guarda los datos y programas que la CPU necesita usar **en ese instante**. | Guarda de manera permanente los datos, el sistema operativo, las aplicaciones y todos tus archivos. |
| **Persistencia** | **Volátil**: se borra cuando el equipo se apaga o reinicia. | **No volátil**: mantiene la información incluso cuando está apagado. |
| **Velocidad** | Muy rápida (GB/s en el rango de 10‑20 GB/s, con latencias 